# Phase 0: Identify Chemotherapy Medications

## Purpose
This notebook identifies and analyzes chemotherapy medications used in Head and Neck Cancer (HNC) patients by:
- Extracting medication codes from PCDM tables
- Translating codes to medication names
- Classifying medications by drug class (e.g., platinum-based, taxanes, etc.)
- Creating visualizations of chemotherapy usage patterns

## Inputs
- Raw PCDM tables:
  - `PCDM_MED_ADMIN` - Medication administration records
  - `PCDM_PRESCRIBING` - Prescription records
  - `PCDM_DEMOGRAPHIC` - Patient demographics
- Translation dictionaries from `input_data/ICD_RX translation/`

## Outputs
- **Visualizations**: Bar charts showing frequency of different chemotherapy agents
- **Analysis tables**: Lists of chemotherapy medications by drug class
- **Summary statistics**: Usage patterns across the cohort

## Key Chemotherapy Classes Identified
- Platinum-based agents (cisplatin, carboplatin)
- Taxanes (paclitaxel, docetaxel)
- 5-Fluorouracil (5-FU)
- Immunotherapy agents (pembrolizumab, nivolumab)
- EGFR inhibitors (cetuximab)
- Other chemotherapy agents

## What This Notebook Does
1. Extracts all medication codes from administration and prescription records
2. Filters for medications known to be chemotherapy agents
3. Translates medication codes to ingredient names
4. Groups medications by therapeutic class
5. Calculates frequency and prevalence across patients
6. Creates visualizations showing usage patterns

## When to Run
- **Exploratory analysis phase** to understand treatment patterns
- To identify which chemotherapy agents are most common
- Before feature engineering to understand which medications to include
- To validate that expected treatments appear in the data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests
import math

# Set pandas view options to display all rows and columns
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


In [ ]:
### check if value is a number
def isNumber(s):
    punc = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
    for ele in s:
        if ele in punc:
            
            s = s.replace(ele, "")
            return True
    try:
        float(s)
        return True
    except:
        return False


### Run this first once to ensure that medadmin has necessary colunn
### Adds medadmin name truncated to medadmin file
# initial words list to split medications
### Takes about:  
def medicationNameListMaker(medadmin, col_name):
    words = []
    [words.append(word.split()) for word in medadmin[col_name]]
    #print(words)
    #create medication list of just the medication
    labels = [''] * len(words)
    i = 0
    print(medadmin[col_name][i])
    for word in words:
        #print(word)
        for part in word:
            #print(part)
            if (isNumber(part)):
                #print(labels[i])
                #print("end")
                break
            labels[i] += part + ' '
        print(labels[i])
        i+=1

    #Remove last space
    medications =  [label[:-1] for label in labels]
    medadmin['Medication_Name'] = medications
    return medications

def is_none_or_whitespace(s):
    return s is None or s == "" or str.lower(s) =='nan'

### Run this first once to ensure that medadmin has necessary colunn
### Adds medadmin name truncated to medadmin file
# initial words list to split medications
### Takes about:  
def medicationNameExtractor(raw_med_name):
    # words = []
    # [words.append(word.split()) for word in medadmin[col_name]]
    #print(words)
    #create medication list of just the medication
    try:
        str(raw_med_name)
    except:
        return ''


    if type(raw_med_name) == float:
        return ''

    if is_none_or_whitespace(raw_med_name):
        return ''
    word = ''
    for part in raw_med_name:
        if (isNumber(part)):
            #print(labels[i])
            #print("end")
            break
        word += part
    #Remove last space
    output =  word[:-1] 
    return str.lower(output)

In [ ]:
PCDM_MED_ADMIN = pd.read_csv('/path/to/IRB_DATA_DRIVE/PCDM_2023_03_09/IRB_PCDM_MED_ADMIN.csv',on_bad_lines='skip' ,encoding = 'latin-1', engine='python')
print("11/17 Read in PCDM_MED_ADMIN")
PCDM_PRESCRIBING = pd.read_csv('/path/to/IRB_DATA_DRIVE/PCDM_2023_03_09/IRB_PCDM_PRESCRIBING.csv',on_bad_lines='skip' ,encoding = 'latin-1', engine='python')
print("14/17 Read in PCDM_PRESCRIBING")

In [ ]:
### Translation dictionary for rxnorm cui codes
column_types = {'RX_CODE':str, 'TRANSLATION':str}
rxCodeTranslatedf = pd.read_csv('input_data/ICD_RX translation/rxCodeTranslationDF_HNC.csv', dtype=column_types)
rxCodeTranslatedf['TRANSLATION'] = rxCodeTranslatedf['TRANSLATION'].fillna('')
rxCodeTranslateDict =dict(zip(rxCodeTranslatedf['RX_CODE'], rxCodeTranslatedf['TRANSLATION']))

### Translation dictionary for NDC med codes
column_types = {'NDC_CODE':str, 'TRANSLATION':str}
ndcTranslatedf = pd.read_csv('input_data/ICD_RX translation/ndcCodeTranslationDF_HNC.csv', dtype=column_types)
ndcTranslatedf['TRANSLATION'] = ndcTranslatedf['TRANSLATION'].fillna('')
ndcCodeTranslateDict =dict(zip(ndcTranslatedf['NDC_CODE'], ndcTranslatedf['TRANSLATION']))

In [ ]:
PCDM_MED_ADMIN

In [ ]:
PCDM_PRESCRIBING

### PCDM_PRESCRIBING

In [ ]:
### PREP COLUMNS
PCDM_PRESCRIBING['RXNORM_CUI'] = PCDM_PRESCRIBING['RXNORM_CUI'].fillna(0)
rxCodeTranslateDict['0.0'] = ''
PCDM_PRESCRIBING = PCDM_PRESCRIBING[PCDM_PRESCRIBING['RXNORM_CUI']!='N']

### TRANSLATE MED CODES IN PRESCRIBING
prescribingMedicationRootName = []
for i,val in tqdm(enumerate(PCDM_PRESCRIBING['RXNORM_CUI'])):
    #print(float(val))
    val = str(float(val))
    translation = rxCodeTranslateDict[val]
    if translation == '':
        rawMed_Name = PCDM_PRESCRIBING.loc[i, 'RAW_RX_MED_NAME']
        #print(rawMed_Name)
        prescribingMedicationRootName.append(medicationNameExtractor(rawMed_Name))
    else:
        prescribingMedicationRootName.append(translation)

PCDM_PRESCRIBING['TRANSLATED_MED_CODES'] = prescribingMedicationRootName


In [ ]:
PCDM_PRESCRIBING

In [ ]:
# Group by dx_code and count unique patient_id
result = PCDM_PRESCRIBING.groupby('TRANSLATED_MED_CODES')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['TRANSLATED_MED_CODES', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result[0:25]

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['TRANSLATED_MED_CODES'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()

#### CHEMO

In [ ]:
chemotherapy_medications = ['cisplatin', 'carboplatin', 'paclitaxel', 'docetaxel', '5-fluorouracil', '5-fu', 'fluorouracil','capecitabine', 'gemcitabine', 'doxorubicin', 'adriamycin', 'epirubicin', 'cyclophosphamide', 'cyclophosphamide', 'ifosfamide', 'etoposide', 'irinotecan', 'topotecan', 'vinorelbine', 'vinblastine', 'vincristine', 'bleomycin', 'mitomycin', 'mitoxantrone', 'oxaliplatin', 'pemetrexed', 'docetaxel', 'paclit', 'hydroxyurea', 'taxotere']

In [ ]:
#chemotherapy_medications_additions = ['hydroxyurea', 'toripalimab', 'nivolumab', 'taxotere', 'avastin', 'bevacizumab', 'erbitux', 'cetuximab', 'keytruda', 'opdivo', 'yervoy', 'ipilimumab', 'velcade', 'bortezomib', 'revlimid', 'lenalidomide', 'thalomid', 'thalidomide']

In [ ]:
chemo_therapy_drug_data = PCDM_PRESCRIBING[PCDM_PRESCRIBING['TRANSLATED_MED_CODES'].str.contains('|'.join(chemotherapy_medications), case=False, na=False)] 

In [ ]:
chemo_therapy_drug_data

In [ ]:
# Group by dx_code and count unique patient_id
result = chemo_therapy_drug_data.groupby('TRANSLATED_MED_CODES')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['TRANSLATED_MED_CODES', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['TRANSLATED_MED_CODES'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()

#### Immuno

In [ ]:
immuno_medications = ['pembrolizumab', 'toripalimab', 'nivolumab', 'avastin', 'bevacizumab', 'erbitux', 'cetuximab', 'keytruda', 'opdivo', 'yervoy', 'ipilimumab', 'revlimid', 'lenalidomide', 'thalomid', 'thalidomide']

In [ ]:
immuno_therapy_drug_data = PCDM_PRESCRIBING[PCDM_PRESCRIBING['TRANSLATED_MED_CODES'].str.contains('|'.join(immuno_medications), case=False, na=False)]

In [ ]:
immuno_therapy_drug_data

In [ ]:
# Group by dx_code and count unique patient_id
result = immuno_therapy_drug_data.groupby('TRANSLATED_MED_CODES')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['TRANSLATED_MED_CODES', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['TRANSLATED_MED_CODES'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()

### PCDM_MED_ADMIN

In [ ]:
medicationRootName = []
for i,val in tqdm(enumerate(PCDM_MED_ADMIN['MEDADMIN_CODE'])):
    code_type = PCDM_MED_ADMIN.loc[i, 'MEDADMIN_TYPE']
    #print(i)
    if code_type == 'RX':
        translation = rxCodeTranslateDict[val]
        if translation == '':
            rawMed_Name = PCDM_MED_ADMIN.loc[i, 'RAW_MEDADMIN_MED_NAME']
            medicationRootName.append(medicationNameExtractor(rawMed_Name))
        else:
            medicationRootName.append(translation)
    elif code_type == 'ND':
        translation = ndcCodeTranslateDict[val]
        if translation == '':
            rawMed_Name = PCDM_MED_ADMIN.loc[i, 'RAW_MEDADMIN_MED_NAME']
            medicationRootName.append(medicationNameExtractor(rawMed_Name))
        else:
            medicationRootName.append(translation)
    else:
        rawMed_Name = PCDM_MED_ADMIN.loc[i, 'RAW_MEDADMIN_MED_NAME']
        medicationRootName.append(medicationNameExtractor(rawMed_Name))

PCDM_MED_ADMIN['TRANSLATED_MED_CODES'] = medicationRootName

In [ ]:
# Group by dx_code and count unique patient_id
result = PCDM_MED_ADMIN.groupby('TRANSLATED_MED_CODES')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['TRANSLATED_MED_CODES', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result[0:25]

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['TRANSLATED_MED_CODES'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()

#### Chemotherapy

In [ ]:
chemotherapy_medications = ['cisplatin', 'carboplatin', 'paclitaxel', 'docetaxel', '5-fluorouracil', '5-fu', 'fluorouracil','capecitabine', 'gemcitabine', 'doxorubicin', 'adriamycin', 'epirubicin', 'cyclophosphamide', 'cyclophosphamide', 'ifosfamide', 'etoposide', 'irinotecan', 'topotecan', 'vinorelbine', 'vinblastine', 'vincristine', 'bleomycin', 'mitomycin', 'mitoxantrone', 'oxaliplatin', 'pemetrexed', 'docetaxel', 'paclit', 'hydroxyurea', 'taxotere']

In [ ]:
chemo_therapy_MED_ADMIN_drug_data = PCDM_MED_ADMIN[PCDM_MED_ADMIN['TRANSLATED_MED_CODES'].str.contains('|'.join(chemotherapy_medications), case=False, na=False)]

In [ ]:
len(chemo_therapy_MED_ADMIN_drug_data['PATID'].unique())

In [ ]:
chemo_therapy_MED_ADMIN_drug_data

In [ ]:
# Group by dx_code and count unique patient_id
result = chemo_therapy_MED_ADMIN_drug_data.groupby('TRANSLATED_MED_CODES')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['TRANSLATED_MED_CODES', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['TRANSLATED_MED_CODES'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()

#### Immuno therapy

In [ ]:
immuno_medications = ['pembrolizumab', 'toripalimab', 'nivolumab', 'avastin', 'bevacizumab', 'erbitux', 'cetuximab', 'keytruda', 'opdivo', 'yervoy', 'ipilimumab', 'revlimid', 'lenalidomide', 'thalomid', 'thalidomide']

In [ ]:
immuno_therapy_PCDM_MED_drug_data = PCDM_MED_ADMIN[PCDM_MED_ADMIN['TRANSLATED_MED_CODES'].str.contains('|'.join(immuno_medications), case=False, na=False)]

In [ ]:
immuno_therapy_PCDM_MED_drug_data

In [ ]:
# Group by dx_code and count unique patient_id
result = immuno_therapy_PCDM_MED_drug_data.groupby('TRANSLATED_MED_CODES')['PATID'].nunique().reset_index()

# Rename columns for clarity
result.columns = ['TRANSLATED_MED_CODES', 'unique_patient_count']

result.sort_values(by='unique_patient_count', ascending=False, inplace=True)
# print(result)

In [ ]:
result

In [ ]:
import matplotlib.pyplot as plt
# Plot the data
plot_data = result.head(25)
plt.bar(plot_data['TRANSLATED_MED_CODES'], plot_data['unique_patient_count'])
plt.xlabel('Diagnosis Code')
plt.ylabel('Unique Patient Count')
plt.xticks(rotation=90)
plt.title('Unique Patients Diagnosed with Each Diagnosis Code')
plt.show()